<h1 style="text-align: center;">Telkomsel Exploratory Data Analysis</h1>
<h3 style="text-align: center;">Muhammad Rafi Andrianto</h3>
<h3 style="text-align: center;">Yoanita Dwi Harlandi</h3>

---

## **Section 1. Business Context**

**1.1 Context**

Sebagai operator telekomunikasi terbesar di Indonesia yang mengelola infrastruktur broadband berskala masif, Telkomsel menghadapi perubahan lanskap industri yang dinamis. Pergeseran perilaku konsumen dari layanan legacy (Voice/SMS) menuju layanan Data/Internet memaksa perusahaan terus berinovasi menelurkan berbagai produk kuota data guna mengakuisisi pengguna prabayar baru secara masif melalui aplikasi MyTelkomsel maupun jaringan konter tradisional (Digipos).

Namun, persaingan harga yang sangat agresif (price war) di pasar seluler Indonesia melahirkan tantangan internal yang serius:

1. Fenomena **"Bakar Perdana"**: Paket kuota promo pada kartu perdana baru sengaja di-subsidi agar harganya jauh lebih murah per Gigabyte dibandingkan biaya perpanjangan (top-up) paket normal. Alih-alih melakukan loyalitas pengisian ulang, banyak konsumen memilih membuang kartu SIM lama mereka setelah kuotanya habis dan membeli kartu perdana baru. Hal ini menciptakan metrik semu di mana pertumbuhan pelanggan baru terlihat sangat tinggi (Gross Additions), namun diiringi oleh tingkat kerontokan pelanggan yang luar biasa besar (Churn Rate). Akibatnya, nilai rata-rata pendapatan per pengguna atau ARPU (Average Revenue Per User) perusahaan mengalami penurunan tajam.

2. **Abuse Program Device Bundling**: Kebocoran anggaran subsidi juga terdeteksi pada program kerjasama perangkat premium (misal: beli smartphone gratis kuota 1 tahun). Terdapat indikasi sindikat fraud yang membeli perangkat bundling tersebut untuk diambil kartu SIM subsidi kuotanya demi dijual eceran (dimanfaatkan pada perangkat modem atau HP lain), sedangkan unit smartphone-nya dijual terpisah. Hal ini membuat anggaran subsidi bernilai miliaran rupiah melayang tanpa menghasilkan pelanggan bernilai tinggi (high-value subscribers) yang diharapkan.

Melalui pendekatan berbasis data, tim Data Analytics dituntut untuk melakukan analisis forensik terhadap log aktivitas penggunaan data (Call Data Record - CDR), pelacakan nomor identitas perangkat (IMEI), dan status kepatuhan registrasi prabayar. Langkah ini mendesak dilakukan agar Telkomsel dapat mengubah strategi bisnis dari yang semula berfokus pada kuantitas (mengejar jumlah pengguna baru) menjadi berfokus pada kualitas (retensi pelanggan setia yang profitabel).

**1.2 Problem Statements**

Masalah utama yang dihadapi perusahaan adalah merosotnya nilai ARPU (Average Revenue Per User) serta terjadinya kebocoran anggaran subsidi akibat praktik fraud "Bakar Perdana" dan manipulasi program Device Bundling (IMEI Mismatch).

Secara operasional dan teknis data, masalah tersebut dirumuskan ke dalam beberapa pertanyaan turunan berikut:
1. **Analisis Churn & Regional**: Di area atau regional mana tingkat peredaran kartu "Bakar Perdana" (terindikasi dari tingginya angka churn di bulan pertama) berada pada level tertinggi?

2. **Estimasi Kebocoran Anggaran**: Berapa besar kerugian finansial yang dialami perusahaan akibat penyalahgunaan paket bundling premium di mana identitas perangkat riil yang digunakan berbeda dengan perangkat yang didaftarkan (registered_imei $\neq$ used_imei)?

3. **Standarisasi Infrastruktur Data**: Seberapa besar tingkat inkonsistensi penulisan tipe jaringan (network_type) pada log BTS yang masuk ke database pusat, yang berpotensi memicu bias pada visualisasi performa jaringan broadband?

4. **Risiko Kepatuhan Regulasi (Compliance Risk)**: Berapa persentase nomor pelanggan yang tidak memiliki kejelasan status registrasi NIK/KK, yang menempatkan perusahaan pada risiko denda audit dari Kominfo?

5. **Anomali Sistem & Sinkronisasi**: Berapa banyak sesi penggunaan internet yang bocor secara ilegal sebelum kartu SIM resmi diaktivasi (session_time < activation_date) akibat ketidaksinkronan sistem HLR dan Billing?
    - Berapa banyak temuan outlier pemakaian data ekstrem (nilai fiktif bug billing) yang dapat merusak analisis rata-rata konsumsi data riil pelanggan?


**1.3 Key Objective**

Tujuan utama dari projek analisis ini adalah memberikan rekomendasi strategis berbasis data untuk menghentikan kebocoran pendapatan perusahaan, dengan sasaran spesifik sebagai berikut:

- Mitigasi Churn & Fraud: Mengidentifikasi pola dan wilayah penyebaran kartu "Bakar Perdana" guna merancang kebijakan pembatasan distribusi atau blacklisting ID Outlet konter yang memfasilitasi registrasi massal ilegal.

- Perlindungan Subsidi (Subsidy Protection): Menyediakan basis data untuk penerapan sistem IMEI Locking, sehingga kuota subsidi otomatis hangus jika kartu SIM dipindahkan ke perangkat non-mitra.

- Kepatuhan Sistem (Regulatory Compliance): Mengisolasi nomor-nomor dengan status NIK/KK kosong agar segera dilakukan pemblokiran massal guna mematuhi regulasi Kominfo dan menghindari denda eksternal.

- Integritas Data & Perbaikan Sistem: Menyediakan laporan bug (System Bug Report) bagi tim Data Engineering dan Network Operations untuk menutup celah kebocoran kuota pada database HLR serta membersihkan anomali pencatatan sistem billing.

## **Section 2. Data Understanding**

**2.1 General Information**

In [ ]:
import pandas as pd
import numpy as np

import scipy.stats as stats

import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sub_raw = pd.read_csv('../data/raw/tsel_subscribers.csv')
pkg_raw = pd.read_csv('../data/raw/tsel_packages.csv')
usage_raw = pd.read_csv('../data/raw/tsel_data_usage.csv')

print(f"Subscribers Shape : {sub_raw.shape}")
print(f"Packages Shape    : {pkg_raw.shape}")
print(f"Data Usage Shape  : {usage_raw.shape}")

- tsel_subscribers.csv: Terdiri dari 35.000 baris dan 4 kolom (Master data profil pelanggan)
- tsel_packages.csv: Terdiri dari 10 baris dan 3 kolom (Katalog produk paket data)
- tsel_data_usage.csv: Terdiri dari 300.000 baris dan 7 kolom (Log aktivitas Call Data Record / CDR)

**2.2 Feature Information**
 
**2.2.1 tsel_subscribers.csv**

| Nama Kolom | Deskripsi |
| :--- | :--- |
| **msisdn** | Nomor HP pelanggan (MSISDN). |
| **activation_date** | Tanggal kartu SIM diaktifkan/registrasi. |
| **nik_kk_status** | Status registrasi NIK/KK. |
| **registered_imei** | IMEI smartphone yang didaftarkan saat beli paket bundling (Bisa NaN jika bukan paket bundling). |


**2.2.2 tsel_packages.csv**


| Nama Kolom | Deskripsi |
| :--- | :--- |
| **package_code** | Kode unik paket. |
| **package_name** | Nama paket komersial (Promo Perdana, Halo Bundling, OMG!, Kuota Ketengan). |
| **price** | Harga paket. |

**2.2.3 tsel_data_usage.csv**

| Nama Kolom | Deskripsi |
| :--- | :--- |
| **session_id** | ID unik sesi internet. |
| **msisdn** | Nomor HP pengguna. |
| **package_code** | Kode paket yang memotong kuota. |
| **used_imei** | IMEI HP yang dipakai saat internetan. |
| **session_time** | Tanggal & Jam pengguna menyalakan koneksi data. |
| **network_type** | Jaringan yang dipakai. |
| **payload_mb** | Jumlah Megabyte (MB) yang disedot dalam sesi tersebut. |



In [ ]:
print(f"--- subscribers data type ---\n{sub_raw.dtypes}\n")
print(f"--- package data type ---\n{pkg_raw.dtypes}\n")
print(f"--- usage data type ---\n{usage_raw.dtypes}")

**2.3 Statistics Summary**

In [ ]:
usage_raw['payload_mb'].describe()

Evaluasi metrik statistik deskriptif pada kolom inti payload_mb menunjukkan anomali matematika:

- Penyimpangan Rata2 (Skewness Ekstrem): Nilai rata2 (mean) penggunaan internet berada di angka 6.159,14 MB, sangat jauh melompati nilai tengahnya (median/50%) yang hanya berada di angka 1.024,02 MB. Hal ini membuktikan adanya tarikan dari nilai outliers yang sangat extreme.

- Indikasi Error Sistem: Nilai minimum berada pada angka -150.50 MB, sebuah nilai negatif yang tidak masuk akal untuk kuota pemakaian data fisik

- Nilai maksimum berada pada angka 999.999,90 MB (~1 Terabyte), nilai yang di luar normal

## **Section 3. Data Cleaning**

Tahap ini untuk mengidentifikasi cacat log data seperti anomali, nilai kosong, inkonsistensi format, dan variabel duplikat. Langkah ini menjamin bahwa pengolahan data pada tahap berikutnya menghasilkan visualisasi dan metrik2 yang akurat tanpa bias akibat kerusakan data/sistem

**3.1 Missing Values**

In [ ]:
print("--- tsel_subscribers ---")
print(sub_raw.isna().sum())

print("\n--- tsel_data_usage ---")
print(usage_raw.isna().sum())

invalid_nik = sub_raw[sub_raw['nik_kk_status'].isna()]
pct_invalid_nik = (len(invalid_nik) / len(sub_raw)) * 100

print(f"\nPersentase NIK/KK Status bermasalah/missing: {pct_invalid_nik:.2f}%")

- **nik_kk_status (5202)**: Dari perspektif hukum dan kepatuhan (regulatory compliance), ketiadaan status registrasi sebesar 14,86% merupakan sinyal bahaya risiko denda audit dari Kominfo. Baris ini tidak boleh dihapus dari basis data historis karena nomor-nomor ini harus diisolasi untuk dilaporkan ke tim Fraud & Security guna dilakukan pemblokiran massal.

- **registered_imei (24470)**: Berdasarkan analisis konteks bisnis, ketiadaan data IMEI terdaftar ini bersifat normal dan valid. Pelanggan prabayar reguler yang membeli kartu perdana biasa (non-bundling smartphone) memang tidak meregistrasikan IMEI perangkat mereka saat aktivasi. Mereka membeli kartu SIM secara terpisah untuk dimasukkan ke HP apa saja yang sudah mereka miliki 

**3.2 Duplicated Values**

In [ ]:
print(f"Duplikat di tsel_subscribers : {sub_raw.duplicated().sum()}")
print(f"Duplikat di tsel_packages    : {pkg_raw.duplicated().sum()}")
print(f"Duplikat di tsel_data_usage  : {usage_raw.duplicated().sum()}")

Hasil pengujian mengonfirmasi tidak ada record yang terduplikasi di seluruh baris tabel. Hal ini membuktikan bahwa struktur keunikan data Primary Key (msisdn, package_code, session_id) berada dalam kondisi aman dan terbebas dari redundansi sistem pencatatan log.

**3.3 Identify Spelling Errors**

In [ ]:
usage_raw['network_type'].value_counts()

bisa dilihat adanya inkonsistensi penulisan teks yang terjadi akibat variasi format pengiriman log teknis dari mesin vendor tower BTS yang berbeda2 ke database pusat. Jika inkonsistensi ini tidak ditangani, hasil agregasi performa jaringan broadband akan pecah menjadi 10 bagian terpisah (misalnya performa 4G seolah berbeda dengan LTE atau 4g), sehingga visualisasi data menjadi bias dan dapat menyesatkan stakeholders

**3.4 Identify Anomaly Values**

In [ ]:
sub_dt = sub_raw[['msisdn', 'activation_date']].copy()
sub_dt['activation_date'] = pd.to_datetime(sub_dt['activation_date'])

usage_dt = usage_raw[['msisdn', 'session_time', 'payload_mb', 'session_id']].copy()
usage_dt['session_time'] = pd.to_datetime(usage_dt['session_time'])

#merge untuk validasi tanggal aktivasi dengan tanggal session penggunaan
df_merge_time = pd.merge(usage_dt, sub_dt, on='msisdn', how='left')
anomaly_time = df_merge_time[df_merge_time['session_time'] < df_merge_time['activation_date']]

print(f"jumlah sesi internet SEBELUM tanggal aktivasi: {len(anomaly_time)}")
anomaly_time.head()

Secara logika bisnis prabayar, aktivitas internetan sebelum registrasi kartu SIM adalah kegagalan sistem yang mustahil (kebocoran kuota gratis). Hal ini mengonfirmasi adanya bug sinkronisasi antara database dengan sistem penagihan.

In [ ]:
outlier_extreme = usage_raw[usage_raw['payload_mb'] >= 50000] # max sementara 50 GB per sesi
outlier_extreme_min = usage_raw[usage_raw['payload_mb'] < 0] 
print(f"\njumlah sesi yang melebihi 50,000 MB: {len(outlier_extreme)}")
print(f"jumlah sesi yang dibawah dari 0 MB (negatif): {len(outlier_extreme_min)}\n")

print("top nilai terbesar payload_mb:")
print(usage_raw['payload_mb'].nlargest())

print("\ntop nilai negatif payload_mb:")
print(usage_raw['payload_mb'].nsmallest())

nilai 999.999,9 MB (~1 Terabyte dalam satu sesi) muncul sebanyak 1.544 kali dan nilai -150,5 MB muncul sebanyak 1469 kali. Jika dipertahankan, nilai korup ini mendistorsi nilai rata2 (mean) penggunaan data seperti yang dari seharusnya ~1 GB melonjak ke 6,1 GB, sehingga merusak pemodelan kapasitas jaringan.

**3.5 Creating Clean Datasets**

In [ ]:
sub_clean = sub_raw.copy()
pkg_clean = pkg_raw.copy()
usage_clean = usage_raw.copy()

sub_clean['activation_date'] = pd.to_datetime(sub_clean['activation_date'])
sub_clean['registered_imei'] = sub_clean['registered_imei'].astype('Int64')
usage_clean['session_time'] = pd.to_datetime(usage_clean['session_time'])

In [ ]:
sub_clean.dtypes

In [ ]:
#Tangani Missing Value pada NIK/KK Status (Ubah NaN jadi missing/unregis)
sub_clean['nik_kk_status'] = sub_clean['nik_kk_status'].fillna('Missing/Unregistered')

In [ ]:
#Standardisasi Network Type (mapping 10 variasi menjadi 3)
network_map = {
    '4G': '4G', 
    'LTE': '4G', 
    '4G LTE': '4G', 
    '4g': '4G',
    '5G': '5G', 
    'NR': '5G', 
    '5g': '5G',
    'WCDMA': '3G', 
    '3G': '3G', 
    'HSDPA': '3G'
}
usage_clean['network_type'] = usage_clean['network_type'].map(network_map)
usage_clean['network_type'].value_counts()

In [ ]:
#Hapus outlier dan value aneh pada Payload MB (hapus nilai < 0 dan nilai extreme 999999.9)
usage_clean = usage_clean[(usage_clean['payload_mb'] >= 0) & (usage_clean['payload_mb'] < 999999.9)]

#Hapus sesi internet yang dipakai sebelum tanggal aktivasi kartu
#merge sementara dengan data activation_date untuk memfilter baris yang valid
df_merge_filter = pd.merge(usage_clean, sub_clean[['msisdn', 'activation_date']], on='msisdn', how='left')
usage_clean = df_merge_filter[df_merge_filter['session_time'] >= df_merge_filter['activation_date']].drop(columns=['activation_date'])

In [ ]:
usage_clean['payload_mb'].describe()

In [ ]:
#save to csv
sub_clean.to_csv('../data/cleaned/clean_subscribers.csv', index=False)
usage_clean.to_csv('../data/cleaned/clean_data_usage.csv', index=False)
pkg_raw.to_csv('../data/cleaned/clean_package.csv', index=False)

## **Section 4. Analytics**

In [ ]:
df_sub = pd.read_csv('../data/cleaned/clean_subscribers.csv')
df_usage = pd.read_csv('../data/cleaned/clean_data_usage.csv')
df_pkg = pd.read_csv('../data/cleaned/clean_package.csv')

### **4.1 Analisis Kasus Fraud IMEI Mismatch (Device Bundling Abuse)**

Program bundling premium (seperti Halo Bundling iPhone/Samsung) memiliki beban subsidi harga yang sangat besar dari Telkomsel dengan harapan pengguna akan terus memakai kartu tersebut di smartphone paketan bundling mereka. Jika used_imei (perangkat riil saat internetan) berbeda dengan registered_imei (perangkat saat didaftarkan), ini menjadi bukti bahwa kartu SIM tersebut dicopot untuk digunakan di device lain.

In [ ]:

df_analisis = pd.merge(df_usage, df_sub, on='msisdn', how='inner')
df_analisis = pd.merge(df_analisis, df_pkg, on='package_code', how='inner')
df_analisis.head()

In [ ]:
#Filter hanya untuk pelanggan program bundling (iphone dan samsung)
#Kolom registered_imei tidak kosong pada pelanggan bundling
df_bundling = df_analisis[df_analisis['registered_imei'].notna()].copy()

#tandai
df_bundling['imei_status'] = np.where(df_bundling['used_imei'] == df_bundling['registered_imei'], 'Match', 'Mismatch')
df_bundling.head()

In [ ]:
summary_bundling = df_bundling.groupby('imei_status').agg(
    total_sesi=('session_id', 'count'),
    total_payload_gb=('payload_mb', lambda x: x.sum() / 1024)
).reset_index()

print(summary_bundling)

largest_payload_missmatch = df_bundling[df_bundling['imei_status'] == 'Mismatch'].nlargest(10, 'payload_mb')['payload_mb']
largest_payload_match = df_bundling[df_bundling['imei_status'] == 'Match'].nlargest(10, 'payload_mb')['payload_mb']

print("\nTop 5 Match Payloads (MB):")
print(largest_payload_match.tolist())

print("\nTop 5 Mismatch Payloads (MB):")
print(largest_payload_missmatch.tolist())



Kita akan melakukan Uji Hipotesis Statistik (Independent T-Test) untuk membuktikan: apakah kelompok yang terindikasi IMEI Mismatch mengkonsumsi kuota (payload_mb) lebih banyak dan extreme dibandingkan kelompok IMEI Match.

Hipotesis Uji:
- $H_0$: Rata-rata konsumsi data per sesi antara pengguna IMEI Match dan IMEI Mismatch adalah Sama.
- $H_1$: Rata-rata konsumsi data per sesi pengguna IMEI Mismatch Signifikan Lebih Tinggi (Bukti eksploitasi kuota subsidi).

In [ ]:
#uji hipotesis statistik (Independent T-Test / Welch's T-Test)
match = df_bundling[df_bundling['imei_status'] == 'Match']['payload_mb']
mismatch = df_bundling[df_bundling['imei_status'] == 'Mismatch']['payload_mb']

t_stat, p_value = stats.ttest_ind(mismatch, match, equal_var=False)

print(f"T-Statistic : {t_stat}")
print(f"P-Value     : {p_value}")

if p_value < 0.05:
    print("Kesimpulan  : Tolak H0! Terdapat perbedaan konsumsi data yang SIGNIFIKAN secara statistik")
    print("              Kelompok IMEI Mismatch terbukti menyedot kuota lebih banyak")
else:
    print("Kesimpulan  : Gagal Tolak H0. Tidak ada perbedaan signifikan")

In [ ]:
#visualisasi total kebocoran kuota subsidi

plt.figure(figsize=(8, 6))
sns.barplot(
    data=summary_bundling, 
    x='imei_status', 
    y='total_payload_gb', 
    hue='imei_status'
)

plt.title('Total Kebocoran Kuota Subsidi Program Bundling', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Status IMEI', fontsize=12)
plt.ylabel('Total Volume Data (GB)', fontsize=12)

#menambahkan label angka di atas bar
for p in plt.gca().patches:
    plt.gca().annotate(
        f"{p.get_height():,.2f} GB", 
        (p.get_x() + p.get_width() / 2., p.get_height()), 
        ha='center', va='center', 
        xytext=(0, 8), 
        textcoords='offset points', 
        fontsize=11, 
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

In [ ]:

# visualisasi distribusi pemakaian per sesi
plt.figure(figsize=(8, 6))

sns.boxplot(
    data=df_bundling, 
    x='imei_status', 
    y='payload_mb', 
    hue='imei_status'
)

plt.title('Karakteristik Distribusi Pemakaian per Sesi', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Status IMEI', fontsize=12)
plt.ylabel('Volume per Sesi (MB)', fontsize=12)

plt.tight_layout()
plt.show()

> ini membuktikan bahwa pelaku fraud device bundling bukan tipe pengguna serakah (tethering abuse massal), melainkan kartu SIM premium tersebut sengaja dicopot dan diecer/dijual kembali ke pengguna prabayar reguler dengan pola pemakaian handphone normal (tetap berkisar di angka ~1 GB per sesi). Anggaran subsidi tetap bocor secara finansial karena target mengakuisisi segmen high-value smartphone users gagal

**Recommend:** Manajemen tidak bisa mendeteksi fraud ini hanya dengan memfilter batasan kuota (karena pemakaiannya normal). Salah satu solusi yang dapat diterapkan adalah implementasi **IMEI Locking** secara real-time pada Core Network (HLR/PCRF). Jika sistem mendeteksi `used_imei` $\neq$ `registered_imei`, paket data bundling premium harus otomatis dinonaktifkan atau diblokir seketika.

### **4.2 Korelasi Fenomena "Bakar Perdana" dengan Risiko Regulasi NIK/KK**

In [ ]:
#crosstab frekuensi nilai absolut
cross_nik_abs = pd.crosstab(df_analisis['package_name'], df_analisis['nik_kk_status'])

#crosstab persentase
cross_nik_pct = pd.crosstab(df_analisis['package_name'], df_analisis['nik_kk_status'], normalize='index') * 100

print("=== FREKUENSI ABSOLUT ===")
print(cross_nik_abs)

print("\n=== PROPORSI PERSENTASE (%) ===")
print(cross_nik_pct.round(2))

> Jika kita menduga di awal bahwa status NIK/KK bolong (Missing/Unregistered) hanya menumpuk pada paket murah seperti Promo Perdana (gejala konter nakal meregistrasi massal secara ilegal untuk "Bakar Perdana"), data riil ternyata berkata lain.

Proporsi data yang Missing/Unregistered ternyata terdistribusi sangat merata di kisaran ~14.4% hingga 15.0% di seluruh produk, termasuk pada paket premium sekelas Halo Bundling iPhone (14.74%) dan Internet OMG! 100GB (14.98%). Dari hasil diatas, sebenarnya kita telah dapat jawaban bahwa hilangnya data NIK/KK bukan disebabkan oleh aktivitas kecurangan "Bakar Perdana". Namun, kita bisa membuktikan lagi secara matematis dengan uji Chi-Square.

Uji Chi-Square: Apakah status Missing NIK/KK itu ada hubungannya (dependen) dengan jenis paket tertentu, atau tidak berhubungan sama sekali (independen)?
- $H_0$ (Hipotesis Nol): Status registrasi NIK/KK independen (tidak berhubungan) dengan jenis paket data. (Artinya, bolongnya data NIK/KK murni karena error acak dari sistem internal).
- $H_1$ (Hipotesis Alternatif): Status registrasi NIK/KK dependen (berhubungan/terikat) dengan jenis paket data. (Artinya, ada paket tertentu yang sengaja diincar untuk dieksploitasi tanpa NIK/KK).

In [ ]:

chi2, p_val, dof, expected = stats.chi2_contingency(cross_nik_abs)

print(f"Chi-Square Statistic : {chi2}")
print(f"P-Value              : {p_val}")

if p_value < 0.05:
    print("Kesimpulan           : Tolak H0! Terdapat hubungan antara status registrasi NIK/KK dengan jenis paket data")
else:
    print("Kesimpulan           : Gagal Tolak H0. Tidak ada hubungan")

In [ ]:
# visualisasi
df_vis_nik = cross_nik_pct.loc[:, ['Valid', 'Missing/Unregistered']].sort_values(by='Missing/Unregistered')

sns.set_theme(style="white")
ax = df_vis_nik.plot(
    kind='barh', 
    stacked=True, 
    figsize=(11, 7), 
    color=['#1f77b4', '#d62728'] #biru untuk valid, merah untuk missing
)

plt.title('Proporsi Status Registrasi NIK/KK di Seluruh Lini Produk Paket Data', fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Persentase Kontribusi (%)', fontsize=12)
plt.ylabel('Nama Produk Paket Data', fontsize=12)
plt.legend(title='Status NIK/KK', bbox_to_anchor=(1.02, 1), loc='upper left')

#Menambahkan label
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    x, y = p.get_x(), p.get_y() 
    #hanya beri label jika itu adalah batang 'Missing/Unregistered' (dimulai setelah batang Valid)
    if x > 0 and width > 0:
        ax.annotate(f"{width:.2f}%", 
                    (x + width/2, y + height/2), 
                    ha='center', va='center', 
                    color='white', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

Hasil ini membuktikan bahwa hilangnya data NIK/KK bukan disebabkan oleh aktivitas kecurangan eksternal (sindikat konter nakal). Jika ini ulah sindikat, angka Missing akan melonjak extreme hanya di paket Promo Perdana dan bernilai mendekati 0% di paket premium. Karena polanya seragam di semua paket, ini adalah bukti konkrit adanya kegagalan sistemik internal (Internal System Bug) pada gate/interface integrasi database antara Telkomsel dengan Dukcapil/Kominfo saat proses aktivasi kartu terjadi di seluruh lini produk.

## **Section 5. Conclusion and Recommendation**

**5.1 Conclusion**

In [ ]:
## conclusion

**5.2 Recommendation**